# 一、分词流程

## step1: 准备预料
确认不同类型语料的比例、进行清洗脱敏，再拿出一小部分语料用于之后的验证，训练：验证=99:1

## step2: 预分词阶段（初始化基础单元）
> 设定初始字符/子词单元、简历词汇表雏形

01核心任务是把原始文本切成可统计、可合并的最小单元（不能太碎、也不能太复杂）
在llm的token划分中，常见的策略：
1. 基于规则的预分词（如按空格和标点切分）
2. 按Unicode类别分段（如连续汉字、连续拉丁字母或数字）
3. 基于更底层的UTF-8字节级切分

02对大多数以空格为词边界的语言，可先用正则表达式按单词边界和标点进行初步划分，而对中文、日文等不以空格为词界的语言则通常采用逐字符或基于字的初始单元来保证覆盖性。

03预分词生成的基础单元序列将作为后续统计合并的输入，务必保存该序列与对应位置信息以便在训练过程反复高效更新。

### 基于空格和标点的切分策略

In [6]:
import re

def part(text):
    # 将标点符号单独拆开
    text = re.sub(r'([.,!?;:()"\'\[\]{}])', r' \1 ', text)
    # 按空白切开
    tokens = text.split()
    return tokens

if __name__ == "__main__":
    sample_text = "Hello👋👋,Datawhale成立于2018年!!!"
    tokens = part(sample_text)
    print(tokens)

['Hello👋👋', ',', 'Datawhale成立于2018年', '!', '!', '!']


In [8]:
sample_text = 'Hello, world! '
tokens = part(sample_text)    
print(tokens)

['Hello', ',', 'world', '!']


### 基于UTF-8字节级划分策略（最通用）

In [17]:
def tokenize_byte_level(text):
    tokens = []
    for ch in text:
        utf8_bytes = ch.encode('utf-8')
        # 将每个字节转换为十六进制字符串
        hex_bytes = [f"{b:02x}" for b in utf8_bytes]
        print(f"{ch} 转化为 UTF-8 字节: {hex_bytes}")
        tokens.extend(hex_bytes)
    return tokens

if __name__ == "__main__":
    s = "Hello, world!你"
    print(tokenize_byte_level(s))


H 转化为 UTF-8 字节: ['48']
e 转化为 UTF-8 字节: ['65']
l 转化为 UTF-8 字节: ['6c']
l 转化为 UTF-8 字节: ['6c']
o 转化为 UTF-8 字节: ['6f']
, 转化为 UTF-8 字节: ['2c']
  转化为 UTF-8 字节: ['20']
w 转化为 UTF-8 字节: ['77']
o 转化为 UTF-8 字节: ['6f']
r 转化为 UTF-8 字节: ['72']
l 转化为 UTF-8 字节: ['6c']
d 转化为 UTF-8 字节: ['64']
! 转化为 UTF-8 字节: ['21']
你 转化为 UTF-8 字节: ['e4', 'bd', 'a0']
['48', '65', '6c', '6c', '6f', '2c', '20', '77', '6f', '72', '6c', '64', '21', 'e4', 'bd', 'a0']


In [12]:
utf8_bytes = "你".encode("utf-8")

for b in utf8_bytes:
    print(b)

228
189
160


In [ ]:
for b in utf8_bytes:
    print(f"{b:02x}")
# x表示转成十六进制；2表示至少2位；0表示不够2位就在前面补0

e4
bd
a0


### 预分词保存对应位置信息

In [ ]:
def btp_hex_list(text):
    """"
    UTF-8字节级预分词，返回：
    1. tokens：每个自负的字节序列+位置信息
    2. t: 所有字节的十六进制字符串列表
    """
    tokens =[]
    t = []
    for inx, char in enumerate(text):
        # enumerate()函数遍历 text 的同时，既拿到“位置编号”，又拿到“当前位置的元素”，组成字典{位置编号, 当前位置元素}。
        utf8_bytes = char.encode('utf-8')
        hex_bytes = ' '.join(f"{b:02x}" for b in utf8_bytes)
        tokens.append({
            'char': char,
            'bytes': hex_bytes,
            'start': inx,
            'end': inx + 1
        })
        t.extend([f"{b:02x}" for b in utf8_bytes])
        # append()方法是增加一组元素增加到列表末尾，而extend()方法则是将一个个单一的元素加入
    return tokens, t

if __name__ == "__main__":
    text = "Hi, 你好🐳"
    tokens, hex_list = btp_hex_list(text)
    for i in tokens:
        print(i)
    print(hex_list)

{'char': 'H', 'bytes': '48', 'start': 0, 'end': 1}
{'char': 'i', 'bytes': '69', 'start': 1, 'end': 2}
{'char': ',', 'bytes': '2c', 'start': 2, 'end': 3}
{'char': ' ', 'bytes': '20', 'start': 3, 'end': 4}
{'char': '你', 'bytes': 'e4 bd a0', 'start': 4, 'end': 5}
{'char': '好', 'bytes': 'e5 a5 bd', 'start': 5, 'end': 6}
{'char': '🐳', 'bytes': 'f0 9f 90 b3', 'start': 6, 'end': 7}
['48', '69', '2c', '20', 'e4', 'bd', 'a0', 'e5', 'a5', 'bd', 'f0', '9f', '90', 'b3']


| 方法         | 操作对象 | 作用                   | 示例                           | 结果            |
| ---------- | ---- | -------------------- | ---------------------------- | ------------- |
| `append()` | list | 把一个对象**整体**加到列表末尾    | `[1,2].append([3,4])`        | `[1,2,[3,4]]` |
| `extend()` | list | 把一个可迭代对象**拆开逐个加入**列表 | `[1,2].extend([3,4])`        | `[1,2,3,4]`   |
| `join()`   | str  | 把多个字符串**拼成一个字符串**    | `' '.join(['e4','bd','a0'])` | `'e4 bd a0'`  |


## step3: 统计并迭代更新
> 计算单元共现频率、基于阈值合并高频单元。**先决定“哪些小片段值得成为 token”，再不断调整词表，最终得到一个大小合适、能比较高效表示文本的词表。**

分词器训练的核心是 迭代更新候选子词 -> 控制词表大小或收敛指标 -> 监控质量指标， 不同算法仅在“候选生成方式”和“迭代更新策略”上有差异。

<span style="color:red;">分成两步：子词候选统计、子词候选统计迭代更新。</span>

其中“子词候选统计”是指，遍历语料以收集用于后续决策的统计信息。

包括四种算法：
* BPE：谁经常挨在一起，我就把谁粘起来。属于典型的从小往大建词表：字符/byte -> 小片段 -> 更大片段 -> 常见词/子词
    * 属于“贪心算法”：我不考虑“这样做是不是最终全球最优”，我只选“眼前这一轮最划算的”。
    * 适用：简单、快、非常适合大规模语料
    * <u>关键词：**高频、合并、从小到大、贪心。**</u>
* WordPiece：不仅看“经常一起出现”，还看“是不是特别属于彼此”。与BPE类似，从小往大建词表（从小 token 开始，不断合并成更大的 token。）
    * 公式：$$score(A,B)=\frac{freq(A) \times freq(B)}{freq(AB)}$$
    * 特征：更关注 A和B之间“相见”的次数（如果相较于他们各自出现的次数要多，那么AB会被划为1个token）
    * 适用：更偏向寻找有统计关联性的子词结构，而不只是绝对频率最高。
    * <u>关键词：**关联性、合并、从小到大。**</u>
* Unigram：思路完全反过来——先给你一大堆词，然后不断淘汰废物
    * 特征：最费算力。与BPE、WordPiece相反，属于**从大往小**删减建词表：超大词表 -> 不停删东西 -> 变成目标词表
    * 算法：期望最大化EM。
        * E（期望）：用目前这套词表，看看语料大概都喜欢怎么切。
        * M（最大化）：根据刚才的切分情况，重新调整每个 token 的重要性/概率。
        * 剪枝：不重要的 token 淘汰。
    * <u>关键词：**概率、剪枝、从大到小。**</u>
* SentencePiece：不是像 BPE / WordPiece / Unigram 那样的一套“谁合并谁”的新规则；而是**完整的 tokenizer 工具箱。**

|          | BPE          | WordPiece     | Unigram       | SentencePiece    |
| -------- | ------------ | ------------- | ------------- | ---------------- |
| 核心思路     | 高频 pair 不断合并 | 高关联 pair 不断合并 | 大词表不断淘汰       | tokenizer 工具/框架  |
| 方向       | 从小到大         | 从小到大          | 从大到小          | 内部可用 BPE/Unigram |
| 最核心依据    | 共现频率         | 关联性评分         | token 概率、语料似然 | 取决于内部算法          |
| 最大特点     | 简单、快、成熟      | 比纯频率更讲究组合关系   | 概率建模更灵活       | 可直接处理原始文本、多语言友好  |
| 你现在要掌握程度 | **重点**       | 知道区别          | 理解思路          | 知道它不是第四种算法       |

*补充概念“最大化预料似然”*：让我设计出来的这套 token，尽可能自然、高效地解释现有语料。

## step4: 输出产物并用于编码与解码
> 生成最终分词模型、支持文本编码和解码功能

1. **导出核心产物**

训练完成后都需要导出至少两个关键文件：
* vocab文件：记录所有token及其对应的id。—— 编码器与解码器的索引
* merges文件：按顺序记录所有子词合并规则或概率模型。——具体分词规则

2. <u>**下游使用前，先验证、评估**</u>

建议统计以下关键指标：
* 平均token数与最大长度分布 直接影响显存占用、训练速度和推理效率。
* 碎片化情况 检查关键实体、专业术语是否被拆得过碎，避免影响模型理解。
* 跨语言token平衡度 多语言任务中需确保不同语言的常见模式都有足够的token支持。

# 二、常见分词器

## 字符分词器
> 将文本拆解为最小的字符单位即单个字符形如英语中的字母 （a, b, c） 或者中文里的单字 （你，好）。

* 优点：词表极小、无OOV（能涵盖所有字词）
* 缺点：
    * 序列过长。 一句话变成字符后，长度会增加数倍，大大消耗LLM宝贵的上下文窗口，从而加大LLM的transformer计算显存消耗。
    * 语义稀疏： 单个字符（如t）通常不具备独立的语义，模型需要更深的网络层数来组合出意义。

## 字节分词器
> 核心逻辑是，不再维护“字符”的词表，而是维护一个大小为256的基础词表（0x00到0xFF）

* 应用： 现代LLM如GPT-4, Llama通常不单独使用纯字节分词，而是将字节作为BPE的基础单位即BBPE，这样可以彻底解决跨语言和特殊符号如emoji 🌍等的编码问题。

## 词级分词器
> 在深度学习早期（如RNN时代）这是最主流的方法。它基于空格（英文）或分词算法（中文）将文本切分为具备独立语义的“词”。

* 优点：保留了完整的语义信息比如"apple" 直接对应一个Token ID
* 缺点
    * 词表爆炸： 英语中 look, looks, looked, looking 会被视为4个完全不同的ID
    * OOV 问题严重： 遇到没见过的词如人名、新造词等，只能标记为<UNK>

## 子词级分词器

### BPE分词器（目前主流）
> 统计语料中相邻字符对出现的频率，迭代地将最频繁出现的字符对合并成一个新的Token。

*此外也有Wordpiece、Unigram分词器*

## 总结
| 分词器类型 | 粒度 | 词表大小 | 词表外(OOV) | 序列长度 | 代表模型 |
| :-------- | :--- | :------- | :---------- | :------- | :------- |
| 字符级     | 细   | 小 (100–5k) | 无        | 非常长   | Char‑RNN |
| 字节级     | 更细（字节） | 很小 (~256–1k) | 无    | 很长     | GPT‑2 |
| 词级       | 粗   | 极大 (>100k) | 严重      | 短       | Word2Vec, GloVe |
| BPE        | 中（自适应） | 适中 (30k–100k) | 极少 | 适中     | GPT‑4, Llama 3 |

# 三、总结

<span style="color:red;">tokenizer 拆成三个维度来看</span>
* <u>维度一</u> ：我最开始拿什么当基本积木？
> 对应step1
    * 字符 character
    * 字节 byte
    * 词 word
* <u>维度二</u>：我要不要把这些小积木组合成更好的“子词”？
> 对应step2&3
    * BPE：高频相邻组合不断合并
    * WordPiece：挑关联性更强的组合合并
    * Unigram：先放很多候选，再根据概率不断淘汰
* <u>维度三</u>：整套 tokenizer 用什么工具/框架实现？
> SentencePiece 可以理解成一个“tokenizer 工厂”。

<span style="color:red;">常见 tokenizer</span>

*  字符级 tokenizer
*  字节级 tokenizer
*  词级 tokenizer
* 子词级 tokenizer  
    ├─ BPE  
    ├─ WordPiece  
    └─ Unigram

# 四、应用：分析DeepSeek的分词器

## step1: 加载DeepSeek Tokenizer

In [18]:
from transformers import AutoTokenizer
# 适用DeepSeek Coder 系列模型的分词器
MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"成功加载模型：{MODEL_NAME} 的分词器")
print(f"分词器词表大小V：{len(tokenizer.get_vocab())}")

/opt/anaconda3/envs/llm_py313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


成功加载模型：deepseek-ai/deepseek-coder-6.7b-instruct 的分词器
分词器词表大小V：32022


## step2: DeepSeek分词器的处理逻辑
> DeepSeek优化的字节级BPE词表通过对中文字词分布与代码缩进的精细建模。本文将通过剖析中文分词实例，直观展现其如何通过高频词簇聚合来优化文本序列的效率。

In [21]:
chinese_text = "注意力机制是AI的核心技术。 🚀 🚀"
#编码
encoded_ids = tokenizer.encode(chinese_text, add_special_tokens=False)

#解码回Token字符串（用于观察子集）
#tokenizer内部保存的token形式
tokens = tokenizer.convert_ids_to_tokens(encoded_ids)
# 逐个token ID解码成人能看懂的形式
readable_tokens = [tokenizer.decode([token_id]) for token_id in encoded_ids]

print(f"原始文本: {chinese_text}")
print(f"编码后的Token ID: {encoded_ids}")
print(f"解码后的Token字符串: {tokens}")
print(f"可读的Token字符串: {readable_tokens}")


原始文本: 注意力机制是AI的核心技术。 🚀 🚀
编码后的Token ID: [7421, 1405, 16737, 502, 26888, 337, 12282, 4645, 397, 12394, 235, 209, 12394, 235, 209]
解码后的Token字符串: ['æ³¨æĦı', 'åĬĽ', 'æľºåĪ¶', 'æĺ¯', 'AI', 'çļĦ', 'æł¸å¿ĥ', 'æĬĢæľ¯', 'ãĢĤ', 'ĠðŁ', 'ļ', 'Ģ', 'ĠðŁ', 'ļ', 'Ģ']
可读的Token字符串: ['注意', '力', '机制', '是', 'AI', '的', '核心', '技术', '。', ' �', '�', '�', ' �', '�', '�']


# 五、实践（手搓tokenizer）

## 5.1 照抄目前已有代码
> 目标🎯：搞清楚大致的流程。不用细纠每个模块、方法背后的意思

### step1:分块读取工具函数

In [ ]:
import os
from typing import BinaryIO

def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    在二进制文件中查找分块边界。
    
    参数:
        file: 二进制文件对象。
        desired_num_chunks: 期望的分块数量。
        split_special_token: 用于分割的特殊字节序列。
    """
    # 确保传入的分隔符是字节串类型
    assert isinstance(split_special_token, bytes), "split_special_token must be of type bytes"

    # 获取文件大小
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0) # 回到文件开头

    # 计算每个分块的理想大小
    chunk_size = file_size // desired_num_chunks

    # 初始边界猜测：根据块大小进行均匀分布
    # 边界数组包含起始位置0喝结束为止file_size
    chunk_boundaries = [ i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size  # 确保最后一个边界是文件末尾

    mini_chunk_size = 4096 # 每次向后搜索的缓冲区大小4k字节

    # 遍历除了开头喝结尾外的中间边界点
    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position) #跳转到初步猜测的边界位置

        while True:
            # 读取一个小块数据
            mini_chunk = file.read(mini_chunk_size)

            # 如果读到了文件末尾EOF，说明后面没有分隔符了，直接设为文件末尾
            if mini_chunk ==b"":
                chunk_boundaries[bi] = file_size
                break

            # 在小块中查找分隔符
            found_at = mini_chunk.find(split_special_token)

            if found_at != -1:
                # 找到了分隔符，更新边界位置
                chunk_boundaries[bi] = initial_position + found_at
                break
            else:
                # 没有找到分隔符，继续向后读取下一个小块
                initial_position += mini_chunk_size
    
    return sorted(set(chunk_boundaries))  # 返回排序后的边界列表，去重




### step1: 按块读取文本option2

In [34]:
import time
from tqdm import tqdm

def iter_text_chunks_with_monitor(
    file_path: str,
    chunk_size: int = 1_000_000, # 1MB
    log_every: int = 5,         # 每N个chunk打印一次
):
    start_time = time.time()
    bytes_processed = 0
    chunk_count = 0

    with open(file_path, "r", encoding="utf-8") as f:
        buffer = []
        buffer_size = 0

        for line in f:
            buffer.append(line)
            buffer_size += len(line)
            bytes_processed += len(line)

            if buffer_size >= chunk_size:
                yield "".join(buffer)
                buffer = []
                buffer_size = 0
                chunk_count += 1

                if chunk_count % log_every ==0:
                    log_status(
                        prefix="📘 分词器流式处理",
                        bytes_processed=bytes_processed,
                        start_time=start_time,
                    )
            if buffer:
                yield "".join(buffer)

### 两种分块方式对比

| | `find_chunk_boundaries` | `iter_text_chunks_with_monitor` |
|---|---|---|
| 做什么 | **找切割位置** | **真正读取并产出文本块** |
| 文件读取模式 | 二进制 `bytes` | 文本 `str` |
| 怎么决定边界 | 大致均分后，寻找特殊 token | 一行行累计到指定大小 |
| 是否保护 `<\|endoftext\|>` | **是** | **不专门保护** |
| 是否返回文本 | 否，只返回位置 | 是，`yield` 文本 |
| 是否用于当前训练 | **没有** | **有** |
| 更适合什么 | 并行处理/安全划分独立区间 | 简单流式训练 |

### step2: 训练BPE分词器

#### 先：在不修改tokenizer内部实现的前提下，实时监控内存占用与数据吞吐量，理解tokenizer训练的真实系统行为。

In [35]:
# 当前进程所占内存
import psutil
import os

def get_memory_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024  # 转换为MB

In [36]:
# 日志状态函数输出（内存占用、处理数据量、处理速度）
import time
def log_status(prefix, bytes_processed, start_time):
    elapsed = time.time() - start_time
    mb = bytes_processed / 1024 / 1024
    throughput = mb / elapsed if elapsed > 0 else 0.0 # 计算吞吐量（MB/s）
    mem = get_memory_mb() # 获取当前内存占用

    print(
        f"{prefix} | "
        f"mem={mem:7.1f}MB | "
        f"data={mb:8.1f}MB | "
        f"speed={throughput:6.2f}MB/s | "
    )

#### 再：训练BPE Tokenizer

In [37]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NFKC

> 这是对下面代码的流程分析

原始文本  
    ↓  
NFKC标准化  
    ↓  
ByteLevel预分词  
    ↓  
BPE训练  
    ↓  
生成vocab + merges  
    ↓  
保存tokenizer.json  

第一步，需要创建tokenizer框架  
Tokenizer  
├── normalizer  
├── pre_tokenizer  
├── model  
├── decoder  
└── special tokens  

第二步，Normalizer（分词前进行文本的预处理）  

第三步，ByteLevel （把文字都编码成字节级）。这里不是马上解码，只是说明后续如果要解码该如何做。  

第四步，Decoder（对之前的编码进行解码），用ByteLevel规则把token恢复回来  
token id  
↓  
token  
↓  
byte  
↓  
文本   

第五步（核心），**BPE训练配置**  
> 对应的代码：trainer = BpeTrainer(vocab_size=vocab_size,special_tokens=special_tokens,)  

第六步， 训练真正开始
> 对应代码：tokenizer.train_from_iterator(text_iterator(), trainer=trainer)  

第七步，保存结果


|课程概念|代码|
|-|-|
|预处理|`NFKC()`|
|字节级预分词|`ByteLevel()`|
|BPE算法|`BPE()` + `BpeTrainer()`|
|控制词表大小|`vocab_size`|
|保护特殊token|`special_tokens`|
|训练迭代merge|`train_from_iterator()`|
|输出vocab和merges|`tokenizer.save()`|

In [33]:
# 调用BPE算法、配置一个完整的Tokenizer，并完成训练流程
# 创建BPE模型 → 配置ByteLevel → 设置训练参数 → 喂数据 → 保存tokenizer
def train_bpe_tokenizer(
    train_file: str,
    val_file: str | None = None,
    vocab_size: int = 50257,
    num_chunks: int = 8,
    output_dir: str ="./bpe_tokenizer",
):
    os.makedirs(output_dir, exist_ok=True)

    special_tokens = [
        "<| endoftext |>",
        "<| pad |>",
        "<| unk |>",
        "<| bos |>",
        "<| eos |>",
    ]
    # BPE()：BPE（Byte‑Pair‑Encoding）模型，大语言模型最常用的分词算法。
    tokenizer = Tokenizer(BPE(unk_token="<| unk |>"))

    # 分词前，先对原始字符串做标准化处理。NFKC是Unicode标准的兼容分解规范化形式，能将一些字符转换为更标准的形式。
    tokenizer.normalizer = NFKC()

    # 设计GPT-2风格的BPE Tokenizer
    # tokenizer包括：
    # 1. pre_tokenizer：将原始字符串拆分为更小的单元（如字节、字符、子词等），为后续的BPE编码做准备。
    # 2. decoder：给tokenizer安装一个“翻译官”，告诉它未来如何把内部byte表示翻译回人类文本。
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
    tokenizer.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        show_progress=True,
    )
    def text_iterator():
        #训练集
        for chunk in iter_text_chunks_with_monitor(
            train_file,
            chunk_size=1_000_000,
            log_every=20,
        ):
            yield chunk
    
    print("🚀 开始训练BPE Tokenizer ...")
    tokenizer.train_from_iterator(text_iterator(), trainer=trainer)
    print("✅ BPE Tokenizer训练完成！")

    tokenizer.save(os.path.join(output_dir, "tokenizer.json"))
    print(f"💾 分词器已保存至{output_dir}/tokenizer.json")

    return tokenizer

if __name__ =="__main__":
    train_path = "./TinyStoriesV2-GPt4-train.txt"
    val_path = "./TinyStoriesV2-GPt4-valid.txt"
    tokenizer = train_bpe_tokenizer(
        train_file=train_path,
        val_file=val_path,
        vocab_size = 50257,
        num_chunks=16,
        output_dir="./bpe_tokenizer",
    )


🚀 开始训练BPE Tokenizer ...





NameError: name 'log_statues' is not defined